# 🏥 Mini-Project 4: Data Quality Assessment

**Objective**: Create a reusable Python script that takes any dataset and generates a comprehensive Data Quality Report.

**Skills Tested**:
- Writing functions and modular code
- Outlier detection algorithms (Z-Score, IQR)
- Missing value analysis
- Data validation rules

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.max_columns', None)

## 1. Create the Assessment Toolkit
We will write modular functions to test different aspects of data quality.

In [ ]:
def check_missing_values(df):
    """Returns a summary of missing values per column"""
    missing = df.isnull().sum()
    percent = (missing / len(df)) * 100
    
    summary = pd.DataFrame({
        'Missing Count': missing,
        'Missing Percent': percent
    })
    
    # Only return columns with > 0 missing values, sorted highest to lowest
    return summary[summary['Missing Count'] > 0].sort_values('Missing Percent', ascending=False)

def check_duplicates(df):
    """Checks for exact row duplicates"""
    dups = df.duplicated().sum()
    return dups, (dups / len(df)) * 100

def detect_outliers_iqr(df):
    """Returns count of outliers using IQR method for all numeric columns"""
    outlier_counts = {}
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count values outside the bounds
        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
        
        if outliers > 0:
            outlier_counts[col] = outliers
            
    return outlier_counts

## 2. Generate the Full Report
A wrapper function that calls all the individual checks and prints a nice report.

In [ ]:
def generate_quality_report(df, dataset_name="Dataset"):
    print("="*50)
    print(f"🔍 DATA QUALITY REPORT: {dataset_name.upper()}")
    print("="*50)
    
    # 1. Basic Info
    print(f"\n► OVERVIEW")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    
    # 2. Duplicates
    dup_count, dup_pct = check_duplicates(df)
    print(f"\n► DUPLICATES")
    print(f"Duplicate Rows: {dup_count:,} ({dup_pct:.2f}%)")
    if dup_count > 0:
        print("  ⚠️ Action Required: Drop duplicates unless expected.")
        
    # 3. Missing Values
    missing = check_missing_values(df)
    print(f"\n► MISSING VALUES")
    if len(missing) == 0:
        print("No missing values found! 🎉")
    else:
        print(f"{len(missing)} columns have missing data:")
        print(missing.round(2))
        
    # 4. Outliers
    outliers = detect_outliers_iqr(df)
    print(f"\n► OUTLIER DETECTION (IQR Method)")
    if len(outliers) == 0:
        print("No extreme outliers found.")
    else:
        for col, count in outliers.items():
            pct = (count / len(df)) * 100
            print(f"- {col}: {count:,} outliers ({pct:.2f}%)")
            
    print("\n" + "="*50)

## 3. Test the Toolkit on Messy Data

In [ ]:
# Generate a very messy dataset
np.random.seed(42)
df_messy = pd.DataFrame({
    'ID': range(1000),
    'Score': np.random.normal(50, 10, 1000),
    'Age': np.random.randint(18, 65, 1000),
    'Income': np.random.lognormal(10, 1, 1000)
})

# Inject missing values
df_messy.loc[np.random.choice(df_messy.index, 50, replace=False), 'Age'] = np.nan
df_messy.loc[np.random.choice(df_messy.index, 200, replace=False), 'Income'] = np.nan

# Inject duplicates
df_messy = pd.concat([df_messy, df_messy.iloc[0:15]])

# Inject outliers
df_messy.loc[5, 'Score'] = 9999
df_messy.loc[10, 'Score'] = -500

# RUN THE REPORT!
generate_quality_report(df_messy, "Customer Analytics DB")